***Network Anomaly Detection System***

### **Problem Statement:**

The objective of this project is to build and compare supervised and unsupervised machine learning models for detecting anomalous network traffic under a realistic class imbalance scenario (~5% attacks). The system aims to accurately identify rare intrusion attempts while minimizing false positives, and to analyze the performance trade-offs between classification-based and anomaly detection approaches.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder,StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.decomposition import PCA

from sklearn.ensemble import IsolationForest, RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.neighbors import LocalOutlierFactor as LOF

from sklearn.metrics import (
    accuracy_score, classification_report,
    precision_score, recall_score,
    f1_score, roc_auc_score,
    average_precision_score
)

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam

In [2]:
#Load the dataset
df_train=pd.read_csv('/content/Train_data.csv.zip')

In [3]:
df_train.shape

(25192, 42)

In [4]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25192 entries, 0 to 25191
Data columns (total 42 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   duration                     25192 non-null  int64  
 1   protocol_type                25192 non-null  object 
 2   service                      25192 non-null  object 
 3   flag                         25192 non-null  object 
 4   src_bytes                    25192 non-null  int64  
 5   dst_bytes                    25192 non-null  int64  
 6   land                         25192 non-null  int64  
 7   wrong_fragment               25192 non-null  int64  
 8   urgent                       25192 non-null  int64  
 9   hot                          25192 non-null  int64  
 10  num_failed_logins            25192 non-null  int64  
 11  logged_in                    25192 non-null  int64  
 12  num_compromised              25192 non-null  int64  
 13  root_shell      

In [5]:
#check class distribution
df_train['class'].value_counts()

,count
class,
normal,13449
anomaly,11743


In [6]:
#creating unbalanced dataset with 5% anomalies
df_normal = df_train[df_train['class'] == 'normal']
df_anomaly = df_train[df_train['class'] == 'anomaly']

df_anomaly_small = df_anomaly.sample(
    n=int(len(df_normal) * 0.05),
    random_state=42
)

df_imbalanced = pd.concat([df_normal, df_anomaly_small])

df_imbalanced['class'].value_counts()

,count
class,
normal,13449
anomaly,672


In [7]:
#shuffle after combining
df_imbalanced = df_imbalanced.sample(frac=1, random_state=42).reset_index(drop=True)

In [8]:
df_imbalanced.shape

(14121, 42)

In [9]:
X = df_imbalanced.drop("class", axis=1)
y = df_imbalanced["class"]

In [10]:
#training and validation split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [11]:
X_train.shape, y_train.shape, X_val.shape, y_val.shape

((11296, 41), (11296,), (2825, 41), (2825,))

In [12]:
#Onehotencoding
combined = pd.concat([X_train, X_val])
combined = pd.get_dummies(combined)

X_train = combined.iloc[:len(X_train)]
X_val = combined.iloc[len(X_train):]

In [13]:
y_train.unique()

array(['normal', 'anomaly'], dtype=object)

In [14]:
#Manually encoding target variable
y_train=y_train.map({'normal':0,'anomaly':1})
y_val=y_val.map({'normal':0,'anomaly':1})

In [15]:
X_train.shape,X_val.shape

((11296, 111), (2825, 111))

In [16]:
#Scaling
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

In [17]:
#SMOTE: oversampling technique that generates synthetic minority class samples
#to address class imbalance
#SMOTE applied only on training data

smote = SMOTE(random_state=42)

X_train_sm, y_train_sm = smote.fit_resample(X_train_scaled, y_train)

Pipeline 1 (Supervised):

Split
→ Encode
→ Scale
→ SMOTE
→ RF / XGB

*RandomForest Classifier*

In [18]:
rf=RandomForestClassifier(n_estimators=100,random_state=42)
rf.fit(X_train_sm,y_train_sm)

RandomForestClassifier(random_state=42)

In [19]:
rf_pred = rf.predict(X_val_scaled)
rf_prob = rf.predict_proba(X_val_scaled)[:, 1]

#evaluation
accuracy = accuracy_score(y_val, rf_pred)
print(classification_report(y_val, rf_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2673
           1       1.00      0.99      1.00       152

    accuracy                           1.00      2825
   macro avg       1.00      1.00      1.00      2825
weighted avg       1.00      1.00      1.00      2825



*XGBooster Classifier*

In [20]:
xgb=XGBClassifier()
xgb.fit(X_train_sm,y_train_sm)
xgb_pred = xgb.predict(X_val_scaled)
xgb_prob = xgb.predict_proba(X_val_scaled)[:, 1]

In [21]:
#evaluation
accuracy = accuracy_score(y_val, xgb_pred)
print(classification_report(y_val,xgb_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2673
           1       0.99      0.99      0.99       152

    accuracy                           1.00      2825
   macro avg       1.00      1.00      1.00      2825
weighted avg       1.00      1.00      1.00      2825



Pipeline 2(Unsupervised):

Split
→ Encode
→ Scale
→ PCA (only for LOF)
→ Select normal samples
→ Fit IF/LOF/AE
→ Predict on full validation set

*Isolation Forest*

In [22]:
X_train_normal = X_train_scaled[y_train == 0] # only on normal data i.e., y_train=='normal'

isolation_forest = IsolationForest(n_estimators=200,contamination=0.05, random_state=42)
isolation_forest.fit(X_train_normal)

IsolationForest(contamination=0.05, n_estimators=200, random_state=42)

In [23]:
# Anomaly scores (higher = more anomalous)
if_scores = -isolation_forest.decision_function(X_val_scaled)

# Binary predictions
if_pred = isolation_forest.predict(X_val_scaled)
if_pred = np.where(if_pred == -1, 1, 0)

# Average precision
avg_prec = average_precision_score(y_val, if_scores)

print("Average Precision:", avg_prec)
print(classification_report(y_val, if_pred))

Average Precision: 0.9019524346207654
              precision    recall  f1-score   support

           0       1.00      0.95      0.97      2673
           1       0.53      0.93      0.67       152

    accuracy                           0.95      2825
   macro avg       0.76      0.94      0.82      2825
weighted avg       0.97      0.95      0.96      2825



*Local Outlier Factor*


In [24]:
#reduce dimensions for LOF
pca = PCA(n_components=0.95)
# Keep enough principal components to preserve 95% of the total variance in the data.
X_train_pca = pca.fit_transform(X_train_scaled)
X_val_pca = pca.transform(X_val_scaled)

In [25]:
X_train_normal = X_train_pca[y_train == 0]

lof=LOF(n_neighbors=20, contamination=0.05, novelty=True)
lof.fit(X_train_normal)

LocalOutlierFactor(contamination=0.05, novelty=True)

In [26]:
lof_pred = lof.predict(X_val_pca)
lof_pred = np.where(lof_pred == -1, 1, 0)

# Anomaly scores (higher = more anomalous)
lof_scores = -lof.decision_function(X_val_pca)

# Average precision
avg_prec = average_precision_score(y_val, lof_scores)

print("Average Precision:", avg_prec)
print(classification_report(y_val, lof_pred))

Average Precision: 0.6090769166186742
              precision    recall  f1-score   support

           0       1.00      0.95      0.97      2673
           1       0.52      0.92      0.67       152

    accuracy                           0.95      2825
   macro avg       0.76      0.94      0.82      2825
weighted avg       0.97      0.95      0.96      2825



*Autoencoder*


In [27]:
X_train_normal = X_train_scaled[y_train == 0] # only on training set

input_dim = X_train_normal.shape[1]

input_layer = Input(shape=(input_dim,))
encoded = Dense(64, activation="relu")(input_layer)
encoded = Dense(32, activation="relu")(encoded)
decoded = Dense(64, activation="relu")(encoded)
decoded = Dense(input_dim, activation="linear")(decoded)

autoencoder = Model(inputs=input_layer, outputs=decoded)
autoencoder.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="mse"
)

autoencoder.fit(
    X_train_normal,
    X_train_normal,
    epochs=20,
    batch_size=256,
    validation_split=0.2,
    shuffle=True
)
# Reconstruction error on normal training data
recon_train = autoencoder.predict(X_train_normal)
mse_train = np.mean((X_train_normal - recon_train) ** 2, axis=1)

# Choose threshold
threshold = np.percentile(mse_train, 95)

#Evaluation on validation set
recon_val = autoencoder.predict(X_val_scaled)
mse_val = np.mean((X_val_scaled - recon_val) ** 2, axis=1)

ae_pred = (mse_val > threshold).astype(int)

print(classification_report(y_val, ae_pred))

Epoch 1/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - loss: 0.4792 - val_loss: 0.4817
Epoch 2/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.4389 - val_loss: 0.3733
Epoch 3/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.3616 - val_loss: 0.2955
Epoch 4/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.2560 - val_loss: 0.2501
Epoch 5/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.2352 - val_loss: 0.2162
Epoch 6/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.1982 - val_loss: 0.1926
Epoch 7/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.1953 - val_loss: 0.1758
Epoch 8/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.1755 - val_loss: 0.1592
Epoch 9/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.1403 - val_loss: 0.1474
Epoch 10/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.1284 - val_loss: 0.1369
Epoch 11/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.1270 - val_loss: 0.1261
Epoch 12/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.1004 - val_

In [28]:
def evaluate_model(name, y_true, y_pred, scores):
    return {
        "Model": name,
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred),
        "ROC-AUC": roc_auc_score(y_true, scores),
        "PR-AUC": average_precision_score(y_true, scores)
    }

In [30]:
results = []

results.append(evaluate_model("RandomForest", y_val, rf_pred, rf_prob))
results.append(evaluate_model("XGBoost", y_val, xgb_pred, xgb_prob))
results.append(evaluate_model("IsolationForest", y_val, if_pred, if_scores))
results.append(evaluate_model("LOF", y_val, lof_pred, lof_scores))
results.append(evaluate_model("Autoencoder", y_val, ae_pred, mse_val))

results_df = pd.DataFrame(results)
results_df.sort_values(by="PR-AUC", ascending=False)

,Model,Precision,Recall,F1,ROC-AUC,PR-AUC
0,RandomForest,1.000000,0.993421,0.996700,0.999970,0.999482
1,XGBoost,0.993421,0.993421,0.993421,0.999943,0.999135
2,IsolationForest,0.525926,0.934211,0.672986,0.988769,0.901952
4,Autoencoder,0.501742,0.947368,0.656036,0.988533,0.861728
3,LOF,0.524345,0.921053,0.668258,0.972567,0.609077


# Conclusion — Network Intrusion Detection Project

This project compared supervised and unsupervised approaches for detecting anomalous network traffic under a realistic 95–5 class imbalance scenario.


Supervised models significantly outperformed unsupervised methods, achieving near-perfect PR-AUC scores due to their ability to learn explicit decision boundaries from labeled data. Among unsupervised models, IsolationForest showed the strongest performance, while LOF was sensitive to high-dimensional data despite PCA-based reduction.

Overall, the results demonstrate that supervised models are best when labeled attack data is available, while unsupervised methods remain valuable for detecting unseen or zero-day anomalies.